In [1]:
!pip install -q git+https://github.com/facebookresearch/segment-anything.git
!pip install -q opencv-python pycocotools matplotlib torch torchvision pillow

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogluon-multimodal 1.5.0 requires nvidia-ml-py3<8.0,>=7.352.0, which is not installed.
autogluon-timeseries 1.5.0 requires chronos-forecasting<2.4,>=2.2.2, which is not installed.
autogluon-timeseries 1.5.0 requires einops<1,>=0.7, which is not installed.
autogluon-timeseries 1.5.0 requires peft<0.18,>=0.13.0, which is not installed.
amazon-sagemaker-jupyter-ai-q-developer 1.2.9 requires numpy<=2.0.1, but you have numpy 2.4.4 which is incompatible.
amazon-sagemaker-sql-magic 0.1.4 requires numpy<2, but you have numpy 2.4.4 which is incompatible.
autogluon-common 1.5.0 requires numpy<2.4.0,>=1.25.0, but you have numpy 2.4.4 which is incompatible.
autogluon-core 1.5.0 requires numpy<2.4.0,>=1.25.0, but you have numpy 2.4.4 which is incompatible.
autogluon-features 1.5.0 requires numpy<2.4.0,>=1.25.0, but you have 

In [2]:
import sagemaker


sess = sagemaker.Session()
role_arn = sess.get_caller_identity_arn()
print(f"✓ IAM Execution Role ARN: {role_arn}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
✓ IAM Execution Role ARN: arn:aws:iam::861276118242:role/service-role/AmazonSageMaker-ExecutionRole-20260511T121821


In [3]:
import boto3
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.patheffects as pe 
import cv2
from PIL import Image
import io
import os
import tempfile
import json
from datetime import datetime
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator

In [4]:
S3_BUCKET = 'ztm-sagemaker-course'
S3_IMAGE_KEY = ''
S3_MODEL_BUCKET = 'ztm-sagemaker-course'
S3_MODEL_KEY = 'sam_vit_h_4b8939.pth'
MODEL_TYPE = 'vit_h'
S3_BUCKET_REGION = 'us-west-2'
OUTPUT_PREFIX = 'segmentation_output'

In [5]:
# FILTER: Only process these labels
TARGET_LABELS = {'Car', 'Vehicle', 'Automobile', 'Transportation', 'Person', 'Pedestrian', 'Human'}

In [6]:
# Initialize AWS clients
s3_client = boto3.client('s3', region_name=S3_BUCKET_REGION)
rekognition_client = boto3.client('rekognition', region_name=S3_BUCKET_REGION)

In [7]:
print("✓ Configuration loaded")
print(f"Image: s3://{S3_BUCKET}/{S3_IMAGE_KEY}")
print(f"Model: s3://{S3_MODEL_BUCKET}/{S3_MODEL_KEY}")
print(f"Target labels: {TARGET_LABELS}")

✓ Configuration loaded
Image: s3://ztm-sagemaker-course/
Model: s3://ztm-sagemaker-course/sam_vit_h_4b8939.pth
Target labels: {'Car', 'Vehicle', 'Person', 'Automobile', 'Human', 'Pedestrian', 'Transportation'}


In [8]:
# Helper functions

def load_model_from_s3(bucket, key):
    """Download SAM model from S3 to temporary location"""
    print(f"Loading SAM model from s3://{bucket}/{key}")
    temp_file = tempfile.NamedTemporaryFile(delete=False, suffix='.pth')
    temp_path = temp_file.name
    temp_file.close()

    print("Downloading model...")
    s3_client.download_file(bucket, key, temp_path)
    print(f"✓ Model downloaded")
    return temp_path

def get_image_from_s3(bucket, key, max_size=512):
    """Load image from S3 and convert to RGB"""
    response = s3_client.get_object(Bucket=bucket, Key=key)
    img_data = response['Body'].read()
    img = Image.open(io.BytesIO(img_data))

    if img.mode != 'RGB':
        print(f"  Converting image from {img.mode} to RGB")
        img = img.convert('RGB')

    # Resize if too large (for faster CPU processing)
    if max_size and max(img.size) > max_size:
        ratio = max_size / max(img.size)
        new_size = (int(img.size[0] * ratio), int(img.size[1] * ratio))
        print(f"  Resizing from {img.size} to {new_size} for faster processing")
        img = img.resize(new_size, Image.Resampling.LANCZOS)

    return img

def get_image_size(bucket, key):
    """Get image dimensions from S3"""
    response = s3_client.get_object(Bucket=bucket, Key=key)
    image_data = response['Body'].read()
    image = Image.open(io.BytesIO(image_data))
    if image.mode != 'RGB':
        image = image.convert('RGB')
    return image.size

def euclidean_distance(box1, box2):
    """Calculate Euclidean distance between two bounding boxes"""
    return np.sqrt(sum((a - b) ** 2 for a, b in zip(box1, box2)))

print("✓ Helper functions loaded")

✓ Helper functions loaded


In [9]:
def get_rekognition_detections(bucket, key, target_labels, min_confidence=95):
    """Get object detections from AWS Rekognition - FILTERED by target labels"""
    print(f"Calling AWS Rekognition...")

    response = rekognition_client.detect_labels(
        Image={'S3Object': {'Bucket': bucket, 'Name': key}},
        MaxLabels=100,
        MinConfidence=min_confidence
    )

    width, height = get_image_size(bucket, key)
    print(f"Image dimensions: {width}x{height}")

    # Extract and filter detections
    detections = []
    for label in response['Labels']:
        if label['Name'] in target_labels:
            for instance in label.get('Instances', []):
                box = instance['BoundingBox']
                x = int(box['Left'] * width)
                y = int(box['Top'] * height)
                w = int(box['Width'] * width)
                h = int(box['Height'] * height)

                detections.append({
                    'label': label['Name'],
                    'confidence': instance['Confidence'],
                    'bbox': [x, y, w, h]
                })

    unique_detections = []
    seen_boxes = set()
    for det in detections:
        box_tuple = tuple(det['bbox'])
        if box_tuple not in seen_boxes:
            seen_boxes.add(box_tuple)
            unique_detections.append(det)

    print(f"✓ Found {len(unique_detections)} target objects (cars/pedestrians)")
    return unique_detections

print("✓ Rekognition function loaded")

✓ Rekognition function loaded


In [ ]:
print("\n" + "="*50)
print("INITIALIZING SAM MODEL")
print("="*50)

sam_model_path = load_model_from_s3(S3_MODEL_BUCKET, S3_MODEL_KEY)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

if device == "cpu":
    print("⚠ Running on CPU - using memory-efficient settings")
    torch.set_num_threads(1)

try:
    print(f"Loading SAM model ({MODEL_TYPE})...")
    sam = sam_model_registry[MODEL_TYPE](checkpoint=sam_model_path)
    sam.to(device=device)

    mask_generator = SamAutomaticMaskGenerator(
        sam,
        points_per_side=32,
        pred_iou_thresh=0.88,
        stability_score_thresh=0.95,
        crop_n_layers=1,
        crop_n_points_downscale_factor=2,
        min_mask_region_area=100,
    )

    print("✓ SAM model loaded successfully!")
    os.unlink(sam_model_path)
    print("✓ Temporary model file cleaned up")

except RuntimeError as e:
    print(f"❌ Error: {e}")
    os.unlink(sam_model_path)
    raise

print("="*50)